# BioSkills Lab — Chapter 8
## Logistic Regression and Random Forests on TCGA
[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

> Self-contained. Loads TCGA data automatically if Chapter 7 variables are not in memory.

In [ ]:
!pip install -q scikit-learn numpy pandas matplotlib requests

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, confusion_matrix
print('Libraries loaded')

In [ ]:
try:
    _ = X_train_pca
    print('Chapter 7 data already in memory — skipping download')
except NameError:
    print('Loading TCGA data...')
    BASE = 'https://www.cbioportal.org/api'
    STUDIES = {'BRCA':'brca_tcga','LUAD':'luad_tcga','KIRC':'kirc_tcga','PRAD':'prad_tcga','COAD':'coad_tcga'}
    GENE_IDS = [672,675,7157,4609,5728,1029,595,896,2064,2065,1956,3320,4292,5594,5595,5604,5605,207,208,
                10000,4893,4914,5290,6662,6663,3845,4763,4764,7422,7424,7525,7528,1499,4005,7409,8503,
                9020,2099,2100,2263,2534,3169,3236,4254,4255,5156,5163,5241,5579,6009,6196,7163,7490,
                7849,8379,9054,9500,10257,10319,10488,11200,23533,51176,55294,57142,64919,1026,1027,
                1028,1030,1031,1032,1869,1870,4616,4619,8317,8318,701,994,999,1000,4193,4831,5925,
                6237,7153,7538,23135,3551,3553,3569,3586,3588,3596,6347,6348,6351,6352,6354,6357,
                2246,2247,2248,2251,2252,2253,3908,3909,4318,4320,5265,5266,1278,1280,1281,1282,
                1284,1287,1288,1290,3913,3915,4067,4313,6714,6715,6716,6717,6718,6720,6721,6722,
                6723,6724,6725,6726,2977,2978,2982,2983,2984,2986,2987,2988,2990,1385,1386,4208,
                4602,5062,5159,7040,7042,7043,7046,7048,7049,7422,9369,10912]
    all_frames = []
    for cancer, study in STUDIES.items():
        print(f'  Fetching {cancer}...', end=' ')
        try:
            r = requests.get(f'{BASE}/studies/{study}/samples', params={'pageSize':150}, timeout=30)
            samples = [s['sampleId'] for s in r.json()[:150]]
            r2 = requests.post(f'{BASE}/molecular-profiles/{study}_rna_seq_v2_mrna/molecular-data/fetch',
                               params={'projection':'SUMMARY'},
                               json={'sampleIds': samples, 'entrezGeneIds': GENE_IDS}, timeout=120)
            if r2.status_code != 200: print('failed'); continue
            df = pd.DataFrame([{'sample':d['sampleId'],'gene':d['entrezGeneId'],'value':d['value']}
                               for d in r2.json() if d.get('value') is not None])
            pivot = df.pivot_table(index='sample', columns='gene', values='value', aggfunc='first')
            pivot['cancer_type'] = cancer
            all_frames.append(pivot)
            print(f'{len(pivot)} samples')
        except Exception as e:
            print(f'error: {e}')
    if not all_frames:
        print('API failed — using synthetic data')
        np.random.seed(42)
        n, g = 150, 500
        cancer_types = list(STUDIES.keys())
        blocks, labels_list = [], []
        for i, ct in enumerate(cancer_types):
            X = np.random.randn(n, g)*0.5; X[:,i*100:(i+1)*100]+=3
            blocks.append(X); labels_list.extend([ct]*n)
        expr = pd.DataFrame(np.vstack(blocks), columns=[f'g{i}' for i in range(g)])
        expr['cancer_type'] = labels_list
    else:
        expr = pd.concat(all_frames)
    y_raw = expr['cancer_type'].values
    X_raw = np.nan_to_num(expr.drop(columns=['cancer_type']).values.astype(np.float32))
    le = LabelEncoder()
    y_enc = le.fit_transform(y_raw)
    X_train, X_test, y_train, y_test = train_test_split(X_raw, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)
    pca = PCA(n_components=min(50, X_train_s.shape[1]-1))
    X_train_pca = pca.fit_transform(X_train_s)
    X_test_pca  = pca.transform(X_test_s)
    print(f'Data ready. Shape: {X_train_pca.shape}')

## Logistic Regression

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_train_pca, y_train)
y_pred_lr = lr.predict(X_test_pca)
print(f'LR Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}')
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred_lr)
fig, ax = plt.subplots(figsize=(8,6))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, colorbar=False)
plt.title('Logistic Regression Confusion Matrix')
plt.tight_layout(); plt.show()

## Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300, max_features='sqrt', n_jobs=-1, random_state=42)
rf.fit(X_train_pca, y_train)
y_pred_rf = rf.predict(X_test_pca)
print(f'RF Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}')

In [ ]:
plt.figure(figsize=(10,4))
plt.bar(range(len(rf.feature_importances_)), rf.feature_importances_)
plt.xlabel('Principal Component'); plt.ylabel('Importance')
plt.title('Random Forest Feature Importances')
plt.show()

In [ ]:
print('Method              Accuracy')
print(f'Logistic Regression {accuracy_score(y_test, y_pred_lr):.3f}')
print(f'Random Forest       {accuracy_score(y_test, y_pred_rf):.3f}')